<a href="https://colab.research.google.com/github/adityabadde/PES2UG25CS802-pes-vcs/blob/main/head_impact_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [8]:

# LOAD DATASET


file_path = "/content/dataset.xlsx"

X_train = pd.read_excel(
    file_path,
    sheet_name="Feature Matrix Training"
)

y_train = pd.read_excel(
    file_path,
    sheet_name="Label Vector Training"
)

X_test = pd.read_excel(
    file_path,
    sheet_name="Feature Matrix Testing"
)

y_test = pd.read_excel(
    file_path,
    sheet_name="Label Vector Testing"
)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [9]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (387, 411)
y_train: (387, 1)
X_test: (32, 411)
y_test: (32, 1)


In [10]:

# PREPARE LABELS


y_train = y_train.iloc[:, 0].values
y_test = y_test.iloc[:, 0].values

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining label distribution:")
print(pd.Series(y_train).value_counts())

print("\nTesting label distribution:")
print(pd.Series(y_test).value_counts())

y_train: (387,)
y_test: (32,)

Training label distribution:
0    231
1    156
Name: count, dtype: int64

Testing label distribution:
1    16
0    16
Name: count, dtype: int64


In [11]:

# FEATURE SCALING


from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Learn scaling parameters ONLY from training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the SAME scaling to test data
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (387, 411)
X_test_scaled shape: (32, 411)


In [12]:

# SVM BASELINE MODEL


from sklearn.svm import SVC

# Create SVM classifier
svm_model = SVC(
    kernel="rbf",
    random_state=42
)

# Train the model
svm_model.fit(X_train_scaled, y_train)

# Predict on unseen test data
y_pred_svm = svm_model.predict(X_test_scaled)

print("SVM training and prediction completed!")

SVM training and prediction completed!


In [13]:

# SVM EVALUATION


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_svm)

precision = precision_score(y_test, y_pred_svm)

recall = recall_score(y_test, y_pred_svm)

f1 = f1_score(y_test, y_pred_svm)

cm = confusion_matrix(y_test, y_pred_svm)


# Display results
print("========== SVM RESULTS ==========")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nConfusion Matrix:")
print(cm)

========== SVM RESULTS ==========
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Confusion Matrix:
[[16  0]
 [ 0 16]]


In [14]:

# LOGISTIC REGRESSION BASELINE


from sklearn.linear_model import LogisticRegression

# Create Logistic Regression model
logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

# Train the model
logistic_model.fit(X_train_scaled, y_train)

# Predict on test data
y_pred_logistic = logistic_model.predict(X_test_scaled)

print("Logistic Regression training and prediction completed!")

Logistic Regression training and prediction completed!


In [15]:

# LOGISTIC REGRESSION EVALUATION


accuracy_lr = accuracy_score(y_test, y_pred_logistic)

precision_lr = precision_score(y_test, y_pred_logistic)

recall_lr = recall_score(y_test, y_pred_logistic)

f1_lr = f1_score(y_test, y_pred_logistic)

cm_lr = confusion_matrix(y_test, y_pred_logistic)

print("========== LOGISTIC REGRESSION RESULTS ==========")

print("Accuracy :", accuracy_lr)
print("Precision:", precision_lr)
print("Recall   :", recall_lr)
print("F1 Score :", f1_lr)

print("\nConfusion Matrix:")
print(cm_lr)

========== LOGISTIC REGRESSION RESULTS ==========
Accuracy : 0.90625
Precision: 0.8823529411764706
Recall   : 0.9375
F1 Score : 0.9090909090909091

Confusion Matrix:
[[14  2]
 [ 1 15]]


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Learn mean and standard deviation from training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaling to test data
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (387, 411)
X_test_scaled shape: (32, 411)


In [17]:
# ==========================================
# NEURAL NETWORK MODEL
# ==========================================

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Number of input features
number_of_features = X_train_scaled.shape[1]

print("Number of input features:", number_of_features)


# ------------------------------------------
# CREATE NEURAL NETWORK
# ------------------------------------------

nn_model = Sequential([

    Dense(
        128,
        activation="relu",
        input_shape=(number_of_features,)
    ),

    Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])


# ------------------------------------------
# COMPILE MODEL
# ------------------------------------------

nn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# ------------------------------------------
# SHOW MODEL ARCHITECTURE
# ------------------------------------------

print("\n========== NEURAL NETWORK ARCHITECTURE ==========")

nn_model.summary()


# ------------------------------------------
# TRAIN MODEL
# ------------------------------------------

print("\n========== TRAINING NEURAL NETWORK ==========")

history = nn_model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)


# ------------------------------------------
# PREDICT TEST DATA
# ------------------------------------------

print("\n========== NEURAL NETWORK PREDICTION ==========")

y_probability_nn = nn_model.predict(X_test_scaled)

# Convert probability to class
y_pred_nn = (y_probability_nn >= 0.5).astype(int).ravel()

print("Prediction completed!")


# ------------------------------------------
# EVALUATE MODEL
# ------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

accuracy_nn = accuracy_score(y_test, y_pred_nn)
precision_nn = precision_score(y_test, y_pred_nn)
recall_nn = recall_score(y_test, y_pred_nn)
f1_nn = f1_score(y_test, y_pred_nn)

cm_nn = confusion_matrix(y_test, y_pred_nn)


# ------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------

print("\n========== NEURAL NETWORK RESULTS ==========")

print("Accuracy :", accuracy_nn)
print("Precision:", precision_nn)
print("Recall   :", recall_nn)
print("F1 Score :", f1_nn)

print("\nConfusion Matrix:")
print(cm_nn)

Number of input features: 411

========== NEURAL NETWORK ARCHITECTURE ==========


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        52,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 63,105 (246.50 KB)

 Trainable params: 63,105 (246.50 KB)

 Non-trainable params: 0 (0.00 B)


========== TRAINING NEURAL NETWORK ==========
Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.8123 - loss: 0.4641 - val_accuracy: 0.8846 - val_loss: 0.3401
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9256 - loss: 0.1962 - val_accuracy: 0.7692 - val_loss: 0.3471
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9417 - loss: 0.1787 - val_accuracy: 0.8462 - val_loss: 0.2595
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9515 - loss: 0.1322 - val_accuracy: 0.7692 - val_loss: 0.5515
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9482 - loss: 0.1433 - val_accuracy: 0.7821 - val_loss: 0.3216
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9482 - loss: 0.1278 - val_accuracy: 0.8205 - val_loss: 0.3831
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9709 - loss: 0.0803 - val_accuracy: 0.7821 - val_loss: 0.5360
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9709 -